# Notebook 3 - Hồi quy logistic trên hai bộ feature (Bước 6 - 7)

Notebook này xây mô hình đầu tiên: **hồi quy logistic**, thử trên **hai bộ feature** do Notebook 2 tạo ra:
- **bộ lõi**: bỏ các cột bureau chỉ có dữ liệu từ năm 2012;
- **bộ mở rộng**: có thêm các cột bureau đó.

Câu hỏi cần trả lời bằng số đo: *thêm các cột bureau có làm mô hình tốt hơn không, và hơn bao nhiêu?* Mô hình chỉ cần hơn được **quy tắc đang dùng sẵn của Lending Club** thì mới có giá trị.

**Quy tắc làm việc:** fit trên `train`, đánh giá trên `val`. **Tập `test` chưa được mở** (notebook này không nạp bất kỳ file test nào). Mỗi hàm, mỗi kết quả một ô code; mọi nhận xét do code in ra từ số liệu.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

In [ ]:
SEED = 42
ROOT = Path("..")
DATA_DIR = ROOT / "data/processed"
ART_DIR = ROOT / "artifacts"
FIG_DIR = ART_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

## Nạp dữ liệu (chỉ train và val)

In [ ]:
feature_sets = json.loads((DATA_DIR / "feature_sets.json").read_text())
late_columns = feature_sets["late_columns"]
print("Số cột bureau 'bắt đầu có dữ liệu muộn':", len(late_columns))

In [ ]:
X = {}
for name in ["core", "extended"]:
    X[name] = {"train": np.load(DATA_DIR / f"{name}_train.npy"), "val": np.load(DATA_DIR / f"{name}_val.npy")}
y_train = np.load(DATA_DIR / "y_train.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
for name in X:
    print(f"{name:9s} train {X[name]['train'].shape}, val {X[name]['val'].shape}")

In [ ]:
val_raw = pd.read_parquet(DATA_DIR / "val.parquet", columns=["sub_grade", "int_rate", "default", "term_months", "issue_date"])
assert len(val_raw) == len(y_val) and (val_raw["default"].to_numpy() == y_val).all()
print("Nhãn của val trong file thô khớp từng dòng với nhãn đã lưu.")
print(f"Tỷ lệ vỡ nợ: train {y_train.mean():.4f}, val {y_val.mean():.4f}")
print("Kỳ hạn có trong val (tháng):", sorted(int(t) for t in val_raw["term_months"].unique()),
      "| val từ", f"{val_raw['issue_date'].min():%Y-%m}", "đến", f"{val_raw['issue_date'].max():%Y-%m}")

---
# Bước 6 - Thước đo và baseline

## 6.1 Hai thước đo dùng trong notebook này
| Thước đo | Ý nghĩa | Dùng khi nào | Hạn chế |
|---|---|---|---|
| **ROC-AUC** | Xác suất một khoản vay vỡ nợ được chấm điểm rủi ro cao hơn một khoản vay trả đủ, chọn ngẫu nhiên. 0,5 là đoán mò, 1,0 là hoàn hảo | So sánh khả năng **xếp hạng** rủi ro, chưa cần chọn ngưỡng | Có thể trông tốt dù mô hình bắt kém lớp hiếm |
| **PR-AUC** | Diện tích dưới đường precision-recall. Mô hình vô dụng có PR-AUC bằng tỷ lệ vỡ nợ | Lớp quan tâm (vỡ nợ) là lớp hiếm | Phụ thuộc vào tỷ lệ vỡ nợ nên khó so sánh giữa các tập khác nhau |

Ta **chưa** dùng accuracy: nếu đa số khoản vay trả đủ thì mô hình "đoán không ai vỡ nợ" đã đúng gần hết mà vô dụng. Ô baseline B0 bên dưới sẽ cho thấy điều đó bằng số.

In [ ]:
def evaluate(y_true, score):
    return {"ROC-AUC": roc_auc_score(y_true, score), "PR-AUC": average_precision_score(y_true, score)}

## 6.2 Ba baseline
| Baseline | Ý tưởng |
|---|---|
| **B0** | Đoán "không ai vỡ nợ" cho mọi khoản vay |
| **B1** | Dùng **sub_grade** của Lending Club (A1 = 1, ..., G5 = 35) làm điểm rủi ro: đây là quy tắc có sẵn mà mô hình phải vượt |
| **B2** | Dùng **lãi suất** (`int_rate`) làm điểm rủi ro |

In [ ]:
def sub_grade_number(series):
    letter = series.str[0].map(lambda g: ord(g) - ord("A"))
    return letter * 5 + series.str[1].astype(int)

In [ ]:
val_raw["sub_grade_number"] = sub_grade_number(val_raw["sub_grade"])
print("sub_grade -> số: nhỏ nhất", val_raw["sub_grade_number"].min(), "| lớn nhất", val_raw["sub_grade_number"].max())

**Bảng 1.** Các baseline trên tập validation.

In [ ]:
baseline_scores = {
    "B0 - đoán không ai vỡ nợ": np.zeros(len(y_val)),
    "B1 - sub_grade": val_raw["sub_grade_number"].to_numpy(),
    "B2 - int_rate": val_raw["int_rate"].to_numpy(),
}
results = pd.DataFrame({name: evaluate(y_val, score) for name, score in baseline_scores.items()}).T
results

In [ ]:
b0_accuracy = 1 - y_val.mean()
print(f"B0 đoán 'không ai vỡ nợ' đạt accuracy {b0_accuracy:.1%} nhưng không bắt được khoản vỡ nợ nào (ROC-AUC {results.iloc[0, 0]:.2f}): đây là lý do không dùng accuracy.")
print(f"PR-AUC của B0 ({results.iloc[0, 1]:.4f}) bằng đúng tỷ lệ vỡ nợ của val ({y_val.mean():.4f}).")
print(f"B1 (sub_grade) đạt ROC-AUC {results.iloc[1, 0]:.4f}; B2 (int_rate) đạt {results.iloc[2, 0]:.4f}.")
print("Hành động -> mức mô hình phải vượt là B1; kết quả của B0 chỉ để thấy giá trị đoán mò.")

---
# Bước 7 - Hồi quy logistic trên hai bộ feature

**Tham khảo cách làm của người khác** (theo tên và mô tả project, chưa đối chiếu nội dung gốc):
- Tanish-Srivastava, `credit-risk-scorecard` - https://github.com/Tanish-Srivastava/credit-risk-scorecard : dùng hồi quy logistic và so với hạng (grade) của Lending Club.
- vaibhavkev, `credit-risk` - https://github.com/vaibhavkev/credit-risk : đo mức cải thiện (lift) so với sub_grade trên tập validation theo thời gian.

Hồi quy logistic cho ra một **xác suất vỡ nợ** từ tổng có trọng số của các feature. Cột số đã được chuẩn hóa ở Notebook 2 nên độ lớn hệ số của các cột số so sánh được với nhau.

In [ ]:
def fit_logistic(X_train, y_train):
    model = LogisticRegression(max_iter=1000, random_state=SEED)
    return model.fit(X_train, y_train)

In [ ]:
models = {}
t0 = time.time()
models["core"] = fit_logistic(X["core"]["train"], y_train)
print(f"Bộ lõi: fit xong sau {time.time() - t0:.0f}s, {models['core'].n_iter_[0]} vòng lặp")

In [ ]:
t0 = time.time()
models["extended"] = fit_logistic(X["extended"]["train"], y_train)
print(f"Bộ mở rộng: fit xong sau {time.time() - t0:.0f}s, {models['extended'].n_iter_[0]} vòng lặp")

In [ ]:
val_scores = {name: models[name].predict_proba(X[name]["val"])[:, 1] for name in models}
train_scores = {name: models[name].predict_proba(X[name]["train"])[:, 1] for name in models}
print("Đã tính xác suất dự đoán trên train và val cho cả hai mô hình.")

**Bảng 2.** Các baseline và hai mô hình logistic trên tập validation.

In [ ]:
results.loc["Logistic - bộ lõi"] = pd.Series(evaluate(y_val, val_scores["core"]))
results.loc["Logistic - bộ mở rộng"] = pd.Series(evaluate(y_val, val_scores["extended"]))
results

In [ ]:
auc = results["ROC-AUC"]
b1 = auc["B1 - sub_grade"]
for name in ["Logistic - bộ lõi", "Logistic - bộ mở rộng"]:
    print(f"{name}: ROC-AUC {auc[name]:.4f}, hơn B1 (sub_grade) {auc[name] - b1:+.4f}")
diff = auc["Logistic - bộ mở rộng"] - auc["Logistic - bộ lõi"]
better = "mở rộng" if diff > 0 else "lõi"
print(f"Bộ {better} cao hơn: chênh lệch ROC-AUC giữa hai bộ là {diff:+.4f}, chênh lệch PR-AUC là "
      f"{results.loc['Logistic - bộ mở rộng', 'PR-AUC'] - results.loc['Logistic - bộ lõi', 'PR-AUC']:+.4f}.")
print("Lưu ý -> đây là một lần đo trên một tập validation; chưa kiểm tra mức ý nghĩa thống kê của chênh lệch này, "
      "nên chưa kết luận chắc chắn bộ nào tốt hơn.")

## 7.1 Mô hình có học "thuộc" train không?
Nếu AUC trên train cao hơn hẳn AUC trên val, mô hình đã nhớ dữ liệu train thay vì học quy luật.

**Bảng 3.** ROC-AUC trên train so với validation.

In [ ]:
overfit = pd.DataFrame({
    "train": {name: roc_auc_score(y_train, train_scores[name]) for name in models},
    "val": {name: roc_auc_score(y_val, val_scores[name]) for name in models},
})
overfit["train - val"] = overfit["train"] - overfit["val"]
overfit = overfit.rename(index={"core": "lõi", "extended": "mở rộng"})
overfit

In [ ]:
for name, row in overfit.iterrows():
    direction = "cao hơn" if row["train - val"] > 0 else "thấp hơn"
    print(f"Bộ {name}: AUC train {direction} AUC val {abs(row['train - val']):.4f}.")
print("Một mô hình tuyến tính với số cột này ít có khả năng học thuộc; nếu train - val gần 0 hoặc âm thì không có dấu hiệu quá khớp.")

## 7.2 Các hệ số nói gì?
Mỗi feature có một **hệ số**: hệ số dương làm tăng xác suất vỡ nợ, hệ số âm làm giảm. Ta nhìn mô hình bộ mở rộng, và gắn mỗi cột vào nhóm: *bureau* (cột bắt đầu có dữ liệu muộn), *lõi (số)*, hoặc *phân loại* (one-hot, không chuẩn hóa nên độ lớn không so sánh được với cột số).

In [ ]:
def feature_group(name):
    if name.startswith("cat__"):
        return "phân loại"
    return "bureau" if name.replace("num__", "") in late_columns else "lõi (số)"

In [ ]:
coef = pd.Series(models["extended"].coef_[0], index=feature_sets["extended"]["names"])
coef_table = pd.DataFrame({"hệ số": coef, "nhóm": [feature_group(n) for n in coef.index]})
print("Số hệ số:", len(coef_table), "| theo nhóm:", coef_table["nhóm"].value_counts().to_dict())

**Bảng 4.** 10 hệ số dương lớn nhất và 10 hệ số âm lớn nhất của mô hình bộ mở rộng.

In [ ]:
extremes = pd.concat([coef_table.sort_values("hệ số", ascending=False).head(10),
                      coef_table.sort_values("hệ số").head(10)])
extremes

In [ ]:
numeric_only = coef_table[coef_table["nhóm"] != "phân loại"].copy()
numeric_only["độ lớn"] = numeric_only["hệ số"].abs()
share = numeric_only.groupby("nhóm")["độ lớn"].sum() / numeric_only["độ lớn"].sum()
print("Tỷ lệ tổng độ lớn hệ số của các cột số theo nhóm:")
print(share.apply(lambda v: f"{v:.1%}").to_string())
n_by_group = numeric_only["nhóm"].value_counts()
print(f"Số cột: bureau {n_by_group.get('bureau', 0)}, lõi (số) {n_by_group.get('lõi (số)', 0)}.")
top10 = extremes["nhóm"].value_counts().to_dict()
print("Trong 20 hệ số lớn nhất theo hai phía, số cột theo nhóm:", top10)
print("Hành động -> độ lớn hệ số chỉ cho biết mô hình dùng cột nào nhiều, chưa chứng minh cột đó giúp dự đoán; "
      "bằng chứng đó nằm ở Bảng 2 (so AUC hai bộ).")

## 7.3 Đường ROC trên tập validation
**Biểu đồ 1.** Đường ROC của sub_grade (B1) và hai mô hình logistic. Đường càng gần góc trên bên trái càng tốt.

In [ ]:
curves = {"B1 - sub_grade": baseline_scores["B1 - sub_grade"], "Logistic - bộ lõi": val_scores["core"],
          "Logistic - bộ mở rộng": val_scores["extended"]}
plt.figure(figsize=(6, 5.5))
for name, score in curves.items():
    fpr, tpr, _ = roc_curve(y_val, score)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_val, score):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="đoán mò")
plt.xlabel("Tỷ lệ dương tính giả")
plt.ylabel("Tỷ lệ dương tính thật")
plt.title("Đường ROC trên tập validation")
plt.legend()
plt.savefig(FIG_DIR / "02_roc_logistic_val.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fpr_b1, tpr_b1, _ = roc_curve(y_val, curves["B1 - sub_grade"])
fpr_m, tpr_m, _ = roc_curve(y_val, curves["Logistic - bộ mở rộng"])
point = 0.20
tpr_b1_at = np.interp(point, fpr_b1, tpr_b1)
tpr_m_at = np.interp(point, fpr_m, tpr_m)
print(f"Khi chấp nhận cảnh báo nhầm {point:.0%} khoản vay trả đủ: sub_grade bắt được {tpr_b1_at:.1%} khoản vỡ nợ, "
      f"logistic bộ mở rộng bắt được {tpr_m_at:.1%} (chênh {tpr_m_at - tpr_b1_at:+.1%}).")

---
# Tóm tắt và hạn chế

In [ ]:
print("Tóm tắt trên tập validation:")
print(results.round(4).to_string())
print()
print(f"Số feature: bộ lõi {len(feature_sets['core']['names'])} cột, bộ mở rộng {len(feature_sets['extended']['names'])} cột (sau one-hot).")

In [ ]:
print("Hạn chế (đều đo bằng code ở các notebook trước):")
print(f"1. Tập val chỉ gồm khoản vay kỳ hạn {sorted(int(t) for t in val_raw['term_months'].unique())} tháng, phát hành "
      f"{val_raw['issue_date'].min():%Y-%m} đến {val_raw['issue_date'].max():%Y-%m}; train có cả kỳ hạn 60 tháng.")
print("2. Mới đo một lần trên validation; chưa có khoảng tin cậy cho chênh lệch giữa hai bộ feature.")
print("3. Chưa thử mô hình khác, chưa chỉnh tham số, chưa chọn ngưỡng quyết định.")
print("4. Tập test chưa được mở; chỉ mở một lần, sau khi mọi lựa chọn đã chốt.")